# CEG 论文结果包 Colab Pipeline

该 Notebook 用于在图像生成产物已经存在后, 调用仓库脚本完成 attack、真实 detection、fixed-FPR 校准、论文结果包导出和 Google Drive 归档。方法逻辑不写在 Notebook 中。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/RICHAAARC/CEG.git'
REPO_BRANCH = ''
REPO_ROOT = Path('/content/CEG')
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG')
WORKSPACE = DRIVE_ROOT / 'pilot_runs' / 'real_pilot_input_workspace_20260617_034500'

RUN_PIPELINE = True
ALLOW_INCOMPLETE_PACKAGE = True
ALLOW_INVALID_ARCHIVE = True
ATTACK_FAMILIES = 'brightness_contrast,gaussian_noise,rotate,resize,jpeg'
TARGET_FPR = 0.01
ATTESTATION_KEY_ENV = 'CEG_ATTESTATION_KEY'
ATTESTATION_KEY_ID = 'formal_colab_run_key'


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'非 Colab 环境或 Drive 已挂载: {exc}')

if not REPO_ROOT.exists():
    cmd = ['git', 'clone']
    if REPO_BRANCH:
        cmd += ['--branch', REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print('运行:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all', '--prune'], check=True)
    if REPO_BRANCH:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)


In [ ]:
cmd = [
    sys.executable,
    'scripts/run_colab_paper_results_pipeline.py',
    '--workspace', str(WORKSPACE),
    '--drive-root', str(DRIVE_ROOT),
    '--attack-families', ATTACK_FAMILIES,
    '--target-fpr', str(TARGET_FPR),
    '--attestation-key-env', ATTESTATION_KEY_ENV,
    '--attestation-key-id', ATTESTATION_KEY_ID,
]
if ALLOW_INCOMPLETE_PACKAGE:
    cmd.append('--allow-incomplete-package')
if ALLOW_INVALID_ARCHIVE:
    cmd.append('--allow-invalid-archive')
print('运行:', ' '.join(cmd))
if RUN_PIPELINE:
    subprocess.run(cmd, check=True)
else:
    print('RUN_PIPELINE = False, 仅打印命令。')


In [ ]:
paths = {
    'pipeline_manifest': WORKSPACE / 'paper_results_pipeline' / 'colab_paper_results_pipeline_manifest.json',
    'paper_results_package': WORKSPACE / 'paper_results_pipeline' / 'calibrated_paper_results_package' / 'paper_results_package',
    'drive_archives': DRIVE_ROOT / 'package_archives',
}
for name, path in paths.items():
    print(name, '=', path, 'exists=', path.exists())
